4. Extração de características e construção do dataframe
A equipe deverá transformar cada imagem em um vetor numérico de características. Poderão ser exploradas características de:
- Cor: médias, desvios e histogramas dos canais.
- Textura: LBP e medidas derivadas da matriz de coocorrência.
- Forma: área, perímetro, circularidade, proporções e momentos de Hu.
- Gradientes e bordas: HOG, densidade de bordas e orientação dos gradientes.
  A escolha deverá ser justificada em função do problema. A equipe deverá comparar pelo menos duas configurações de características, por exemplo, cor isoladamente e cor combinada com textura.
O dataframe final deverá conter uma linha por imagem, colunas com as características extraídas, uma coluna com a classe e uma coluna com o identificador da imagem. O identificador não deverá ser utilizado como entrada do classificador.

# Etapa 4 — Extração de características e construção do dataframe

## 4.0 - O que esta etapa faz

Um classificador não "enxerga" imagens: ele só entende números. Nesta etapa, cada imagem
vira uma linha de números chamados **características** (*features*). Juntas, as linhas
formam uma tabela (dataframe) que será usada para treinar e testar os classificadores
na Etapa 5.

As características são medidas **apenas dentro da fruta**, usando a máscara da Etapa 3.
Assim, o fundo branco não influencia os números.

Três grupos de características serão extraídos:
- **Cor:** de que cor é a fruta (médias, desvios e histograma).
- **Textura:** como é a superfície (lisa, rugosa, com manchas): LBP e GLCM.
- **Forma:** o formato da fruta (área, circularidade, proporção, momentos de Hu).

**Configurações a comparar** (os grupos ficarão em colunas separadas para facilitar):
- A: só cor
- B: cor + textura
- C: cor + textura + forma

**Tabela final:** uma linha por imagem, com o identificador da imagem (`id_imagem`),
a classe (a resposta correta), o conjunto (treino ou teste) e as características.
O identificador serve apenas para localizar a imagem e **não pode ser usado como
entrada do classificador**.

In [3]:
# ============================================================
# ETAPA 4 - EXTRAÇÃO DE CARACTERÍSTICAS
# 4.0 - Preparação: bibliotecas, funções das etapas 2 e 3 e dados
# ============================================================

import time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# Funções prontas das Etapas 2 e 3 (arquivo preprocessamento.py)
from preprocessamento import carregar_rgb, preprocessar, segmentar, recortar_fruta, normalizar

# scikit-image: usado mais adiante para textura (LBP, GLCM) e HOG
try:
    import skimage
    print("scikit-image instalado, versão", skimage.__version__)
except ImportError:
    print("ATENÇÃO: instale com  pip install scikit-image")

# --- Carregando a seleção da Etapa 1 (25% de cada classe) ---
df_treino = pd.read_csv("treino_reduzido.csv")
df_treino["conjunto"] = "treino"
df_teste = pd.read_csv("teste_reduzido.csv")
df_teste["conjunto"] = "teste"

# Juntamos treino e teste numa tabela só. A coluna "conjunto" lembra a origem
# de cada imagem, para separar de novo na Etapa 5.
df_imagens = pd.concat([df_treino, df_teste]).reset_index(drop=True)

# --- Identificador único da imagem: conjunto/classe/nome do arquivo ---
def criar_id(linha):
    caminho = Path(linha["caminho"])
    return f"{linha['conjunto']}/{linha['classe']}/{caminho.name}"

df_imagens["id_imagem"] = df_imagens.apply(criar_id, axis=1)

print("Total de imagens:", len(df_imagens))
print(df_imagens["conjunto"].value_counts().to_dict())
print("Identificadores únicos?", df_imagens["id_imagem"].is_unique)
display(df_imagens.head())

ModuleNotFoundError: No module named 'preprocessamento'